# OTT Streaming Content Trend Analysis — Netflix Catalog
**Tools:** Python, Pandas, Matplotlib, Seaborn, SQL  
**Business question:** What does the Netflix catalog reveal about content mix, genre supply, release trends, maturity ratings, geography and duration?

> **Scope & limitations:** This project uses the public Netflix Movies and TV Shows catalog snapshot (commonly distributed as `netflix_titles.csv`). It is not a live catalog and does not contain Prime Video or Disney+ Hotstar rows. It records catalog metadata, not viewing hours, completion rates, subscriber demand, licensing cost or title performance. Therefore, recommendations are supply-side hypotheses—not proof of audience preference.

## 1. Setup and data loading
The notebook first looks for `data/netflix_titles.csv`. If it is not present, it downloads the public TidyTuesday copy. If internet access is unavailable, upload `netflix_titles.csv` to the notebook's `data/` folder and rerun this cell.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import urllib.request

BASE = Path("..") if Path("../data").exists() else Path(".")
DATA_DIR = BASE / "data"
OUT_DIR = BASE / "outputs" / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = DATA_DIR / "netflix_titles.csv"
if not csv_path.exists():
    url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2021/2021-04-20/netflix_titles.csv"
    try:
        urllib.request.urlretrieve(url, csv_path)
        print("Downloaded source dataset.")
    except Exception as exc:
        raise FileNotFoundError(
            "Dataset not found and automatic download failed. Place netflix_titles.csv in the data/ folder. "
            f"Download source: {url}"
        ) from exc

df_raw = pd.read_csv(csv_path)
print("Raw shape:", df_raw.shape)
display(df_raw.head())

## 2. Data quality review
We standardize column names, remove exact duplicate records, parse dates, and derive analysis fields. Missing categorical values are retained as `Unknown` where needed rather than silently discarded.

In [ ]:
df = df_raw.copy()
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

required = {"show_id","type","title","country","date_added","release_year","rating","duration","listed_in"}
missing_cols = required - set(df.columns)
if missing_cols:
    raise ValueError(f"Dataset is missing expected columns: {missing_cols}")

quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_pct", ascending=False)
print("Exact duplicate rows:", int(df.duplicated().sum()))
display(quality)

df = df.drop_duplicates().copy()
for c in ["type","title","country","rating","duration","listed_in"]:
    df[c] = df[c].fillna("Unknown").astype(str).str.strip()
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce").astype("Int64")
df["year_added"] = df["date_added"].dt.year.astype("Int64")
df["month_added"] = df["date_added"].dt.month_name()

# Extract movie runtime and TV season count from the shared duration field.
df["duration_value"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)")[0], errors="coerce")
df["movie_minutes"] = np.where(df["type"].eq("Movie"), df["duration_value"], np.nan)
df["tv_seasons"] = np.where(df["type"].eq("TV Show"), df["duration_value"], np.nan)
df["primary_country"] = df["country"].str.split(",").str[0].str.strip()
df["primary_genre"] = df["listed_in"].str.split(",").str[0].str.strip()
df["rating"] = df["rating"].replace({"": "Unknown"})

print("Cleaned shape:", df.shape)
display(df[["title","type","release_year","year_added","rating","duration","primary_country","primary_genre"]].head())

## 3. Catalog overview and content format

In [ ]:
sns.set_theme(style="whitegrid")
type_counts = df["type"].value_counts()
fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(x=type_counts.index, y=type_counts.values, ax=ax)
ax.set(title="Catalog mix: Movies vs TV Shows", xlabel="Content type", ylabel="Titles")
for i,v in enumerate(type_counts.values): ax.text(i,v,f"{v:,}",ha="center",va="bottom")
plt.tight_layout(); plt.savefig(OUT_DIR/"01_catalog_mix.png",dpi=160); plt.show()

## 4. Release-year trend
This chart uses the original release year—not the year the title was added to Netflix. Recent years can be incomplete in a historical snapshot.

In [ ]:
release_counts = df.dropna(subset=["release_year"]).groupby("release_year").size().sort_index()
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(release_counts.index.astype(int), release_counts.values, linewidth=1.8)
ax.set(title="Titles by original release year", xlabel="Release year", ylabel="Number of titles")
ax.set_xlim(max(1920, int(release_counts.index.min())), int(release_counts.index.max()))
plt.tight_layout(); plt.savefig(OUT_DIR/"02_release_year_trend.png",dpi=160); plt.show()

## 5. Genre mix
Multi-genre titles are split into individual genre rows so each listed genre receives one count per title. These are catalog labels, not mutually exclusive categories.

In [ ]:
genre_long = (df.assign(genre=df["listed_in"].str.split(","))
                .explode("genre"))
genre_long["genre"] = genre_long["genre"].str.strip()
genre_counts = genre_long.query("genre != 'Unknown'")["genre"].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10,6))
sns.barplot(y=genre_counts.index, x=genre_counts.values, ax=ax)
ax.set(title="Top 15 listed genres", xlabel="Title-genre associations", ylabel="Genre")
plt.tight_layout(); plt.savefig(OUT_DIR/"03_genre_mix.png",dpi=160); plt.show()

## 6. Content rating profile
Ratings with missing values remain visible as `Unknown`. Rating systems vary by market, so this is a catalog profile rather than a universal age classification comparison.

In [ ]:
rating_counts = df["rating"].value_counts().head(12)
fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(y=rating_counts.index, x=rating_counts.values, ax=ax)
ax.set(title="Most common content ratings", xlabel="Titles", ylabel="Rating")
plt.tight_layout(); plt.savefig(OUT_DIR/"04_rating_profile.png",dpi=160); plt.show()

## 7. Country mix
A title can list more than one production country. The chart counts title-country associations after splitting the field; it should not be interpreted as exclusive country market share.

In [ ]:
country_long = df.assign(country_item=df["country"].str.split(",")).explode("country_item")
country_long["country_item"] = country_long["country_item"].str.strip()
country_counts = country_long.query("country_item != 'Unknown'")["country_item"].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10,6))
sns.barplot(y=country_counts.index, x=country_counts.values, ax=ax)
ax.set(title="Top 15 production countries (multi-country titles counted for each)", xlabel="Title-country associations", ylabel="Country")
plt.tight_layout(); plt.savefig(OUT_DIR/"05_country_mix.png",dpi=160); plt.show()

## 8. Movie runtime and TV season distributions
The `duration` column stores minutes for movies and seasons for TV shows. These measures are analyzed separately and are not directly comparable.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
movie_runtime = df["movie_minutes"].dropna()
tv_seasons = df["tv_seasons"].dropna()
sns.histplot(movie_runtime, bins=30, ax=axes[0])
axes[0].set(title="Movie runtime distribution", xlabel="Minutes", ylabel="Titles")
sns.countplot(x=tv_seasons.clip(upper=8), ax=axes[1])
axes[1].set(title="TV seasons (8 means 8+)", xlabel="Number of seasons", ylabel="Titles")
plt.tight_layout(); plt.savefig(OUT_DIR/"06_duration_distributions.png",dpi=160); plt.show()

## 9. Netflix additions by year and content type
This uses `date_added`, which can be missing. It measures additions represented in this snapshot, not the platform's complete historical acquisition pipeline.

In [ ]:
added = (df.dropna(subset=["year_added"])
           .groupby(["year_added","type"]).size().unstack(fill_value=0).sort_index())
fig, ax = plt.subplots(figsize=(10,4))
added.plot(kind="bar", stacked=True, ax=ax)
ax.set(title="Catalog additions by year and format", xlabel="Year added", ylabel="Titles added")
plt.tight_layout(); plt.savefig(OUT_DIR/"07_additions_by_year.png",dpi=160); plt.show()

## 10. Genre trend over time
The chart tracks the most common genres by original release year. Use it to spot changes in the composition of the available catalog, not to infer demand or success.

In [ ]:
top_genres = genre_counts.head(6).index.tolist()
trend = (genre_long[genre_long["genre"].isin(top_genres)]
         .dropna(subset=["release_year"])
         .groupby(["release_year","genre"]).size().reset_index(name="titles"))
fig, ax = plt.subplots(figsize=(11,5))
sns.lineplot(data=trend, x="release_year", y="titles", hue="genre", ax=ax)
ax.set(title="Release-year trend for leading genres", xlabel="Original release year", ylabel="Title-genre associations")
plt.tight_layout(); plt.savefig(OUT_DIR/"08_genre_trends.png",dpi=160); plt.show()

## 11. Business summary tables
These outputs are useful for a strategy discussion and are exported for review.

In [ ]:
kpis = pd.DataFrame({
    "metric": ["Total titles","Movies","TV Shows","Distinct release years","Countries listed"],
    "value": [
        len(df),
        int((df["type"]=="Movie").sum()),
        int((df["type"]=="TV Show").sum()),
        int(df["release_year"].nunique()),
        int(country_long.query("country_item != 'Unknown'")["country_item"].nunique())
    ]
})
display(kpis)
display(genre_counts.rename_axis("genre").reset_index(name="title_genre_associations"))
display(df.groupby("type").agg(
    titles=("show_id","count"),
    median_duration_value=("duration_value","median")
).reset_index())

df.to_csv(BASE/"data"/"netflix_titles_cleaned.csv", index=False)
kpis.to_csv(BASE/"outputs"/"catalog_kpis.csv", index=False)
genre_counts.rename_axis("genre").reset_index(name="title_genre_associations").to_csv(BASE/"outputs"/"top_genres.csv", index=False)
print("Saved cleaned data and summary tables.")

## 12. Three actionable recommendations (validate with the computed tables)
1. **Use genre supply gaps to shape acquisition tests.** Compare top genre counts with internal watch hours, completion and retention before increasing or reducing investment. A high catalog count alone does not prove audience demand.
2. **Build a localized acquisition shortlist.** Use the country breakdown to identify underrepresented production markets, then validate each candidate market against local subscriber demand, language coverage, rights availability and cost.
3. **Balance formats and runtimes by audience segment.** Use the movie-runtime and TV-season distributions to define a varied slate. Test shorter films, limited series and longer-running formats against completion and return-viewing metrics before scaling.

**Decision rule:** treat this public catalog as a supply-side baseline. Greenlight decisions require first-party performance, audience research, rights economics and competitive data.

## 13. Caveats and next steps
- Snapshot data; not a live or complete current catalog.
- Netflix only; no direct Prime Video or Hotstar comparison.
- No viewership, ratings/reviews, completion, revenue, licensing cost or subscriber data.
- Multi-country and multi-genre titles are counted in each associated category.
- `release_year` and `year_added` answer different questions.
- For a true OTT competitive analysis, add similarly structured, date-stamped catalogs for each platform and normalize genre, country, rating and format taxonomies before comparing.